# GISLR — Engineered-Feature Discriminability (Angles + Kinematics)

**What this notebook does.** A **dataset-stage, no-training** analysis: reduces
every GISLR video's raw `(T, 543, 3)` landmark tensor to a flat, named,
per-video descriptor vector — joint angles, frame-gap-aware velocity/speed,
Savitzky-Golay jitter, rolling variance, plus the existing
mid-shoulder-centered/inter-shoulder-scaled positions and hand/face relational
distances — then asks, per feature, **is this tight within a class and
separated across classes** (a tolerance-band question), not just "how much
does it move" (`motion-energy`) or "how much class information does a
landmark subset carry" (`subset-comparison`, TODO §3.0 — now broken, see
below).

**Pipeline stage vs. diagnostic.** Like `gislr.1.models.landmark-importance.ipynb`,
this is a one-off interpretability track (`sb.recognize.interp.*`), not wired
into `sb.recognize.architectures.ARCHS`/`registry/` — there is no model here,
only descriptors and discriminability scores. TODO §3.4.

**Why not just re-run `subset-comparison.ipynb`?** That notebook reads raw
per-frame parquet from the retired `asl-signs` Kaggle competition via DuckDB —
broken since the 2026-09-16 move to the pre-converted `GISLR_Stratified` npz
dataset (TODO §0.1). This notebook loads npz directly
(`sb.recognize.features.gislr_stratified.load_npz`), which also happens to be
exactly the `(t, 543, 3)` per-frame shape the feature pipeline was asked to
work from.

**Resumability.** Manifest-driven, same pattern as `motion-energy`/
`subset-comparison`: one parquet artifact per unit, written before the unit is
marked `done`, atomic manifest saves, `done` skipped / `failed` retried.

**Artifacts**

| Path | Content |
|---|---|
| `data/cache/gislr/feature_discriminability/scope_a/*.parquet` | Scope A per-video descriptors (3 classes, ~30 videos) |
| `data/cache/gislr/feature_discriminability/scope_b/*.parquet` | Scope B per-video descriptors (`N_SCOPE_B_CLASSES` classes, chunked) |
| `data/cache/gislr/feature_discriminability/scope_b_manifest.json` | Scope B resumable manifest |
| `docs/reports/feature-discriminability.md` | write-up, **after** this notebook runs (not part of this pass) |

**Design decisions vs. the plan** (`sb.recognize.interp.kinematics`/
`discriminability` — see their module docstrings for the full rationale):

- Frames are never dropped mid-sequence — every gap (one undetected landmark
  or a fully-undetected frame) is linearly interpolated over a *contiguous*
  frame range, so a derivative computed across it reads as steady motion
  rather than a jump, and the frame count a video contributes is preserved.
  `valid_frame_frac` records how much of a video was actually observed vs
  interpolated.
- Per-video descriptors are **mean + std over time** per signal — this is
  intentionally a summary-statistics probe (same spirit as
  `subset-comparison`'s per-video descriptors), not a sequence model: it
  ranks *input information content*, not achievable accuracy.
- Global scope (all 94,477 videos) is **not** run here — Scope B (15 sampled
  classes) is the deliverable of this pass; global is filed as a follow-up
  (§5), same staging `subset-comparison` used before committing to its
  ~50-minute global pass.


In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, gislr_dir
from sb.recognize.interp import discriminability as disc
from sb.recognize.interp import kinematics as kin

DATA_DIR = gislr_dir()
FD_CACHE_DIR = CACHE_DIR / "gislr" / "feature_discriminability"
FD_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42                  # every sampling/split step below uses this seed
SAVGOL_WINDOW = 7          # frames -- same smoothing as motion-energy/subset-comparison
SAVGOL_POLYORDER = 2
ROLLING_VAR_WINDOW = 5     # frames -- windowed variance of per-landmark speed
TOLERANCE_K = 1.5          # tolerance-band width: mean +/- K * std
N_SCOPE_A_CLASSES = 3      # scope A: sampled classes for the eyeball check
N_SCOPE_A_VIDEOS = 10      # scope A: videos per sampled class
N_SCOPE_B_CLASSES = 15     # scope B: sampled sign classes
CHUNK_SIZE = 200           # videos per resumable unit in scope B
VAL_FRAC = 0.15            # probe val split

print("DATA_DIR:", DATA_DIR)
print("cache   :", FD_CACHE_DIR.resolve())
print("angle count:", kin.N_ANGLES, "| relational count:", len(kin.RELATIONAL_NAMES))


## 1. Reusable core — loading, resumable driver, descriptor matrix

In [ ]:
# ============================================================
# train.csv + per-video npz path resolution
# ============================================================
train_csv = pd.read_csv(DATA_DIR / "train.csv")
print(f"{len(train_csv):,} videos - {train_csv['sign'].nunique()} signs")


def npz_path(row: pd.Series) -> Path:
    return DATA_DIR / row["npz_relpath"]


def sample_classes(df: pd.DataFrame, n_classes: int, seed: int = SEED) -> list[str]:
    """Deterministic class sample -- same seeded-sort pattern as subset-comparison."""
    signs = sorted(df["sign"].unique())
    rng = np.random.default_rng(seed)
    return sorted(rng.choice(signs, size=n_classes, replace=False).tolist())


def descriptors_for_rows(rows: pd.DataFrame) -> pd.DataFrame:
    """One batch of train.csv rows -> tidy per-video descriptor DataFrame."""
    out = []
    for _, row in rows.iterrows():
        desc = kin.load_video_descriptors(
            npz_path(row),
            savgol_window=SAVGOL_WINDOW,
            savgol_polyorder=SAVGOL_POLYORDER,
            rolling_var_window=ROLLING_VAR_WINDOW,
        )
        desc["uid"] = row["uid"]
        desc["sign"] = row["sign"]
        out.append(desc)
    return pd.DataFrame(out)


In [ ]:
# ============================================================
# Manifest helpers + resumable driver (pattern from motion-energy/subset-comparison)
# ============================================================
def _manifest_path(scope: str) -> Path:
    return FD_CACHE_DIR / f"{scope}_manifest.json"


def _unit_path(scope: str, unit_id: str) -> Path:
    return FD_CACHE_DIR / scope / f"{unit_id}.parquet"


def load_manifest(scope: str) -> dict:
    p = _manifest_path(scope)
    return json.loads(p.read_text()) if p.exists() else {}


def save_manifest(scope: str, manifest: dict) -> None:
    """Atomic write - a crash mid-save can't corrupt the manifest."""
    p = _manifest_path(scope)
    tmp = p.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp, p)


def process_units(scope: str, unit_ids: list, process_fn, desc: str | None = None) -> dict:
    """Run `process_fn(unit_id) -> DataFrame` over units, resumably.

    Artifact written FIRST, unit marked `done` after; `done` skipped, `failed`
    retried; exceptions recorded in the manifest instead of aborting the run.
    """
    (FD_CACHE_DIR / scope).mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(scope)
    n_skip = n_ok = n_fail = 0
    for uid in tqdm([str(u) for u in unit_ids], desc=desc or scope):
        entry = manifest.get(uid)
        if entry and entry["status"] == "done" and Path(entry["artifact"]).exists():
            n_skip += 1
            continue
        try:
            out = process_fn(uid)
            artifact = _unit_path(scope, uid)
            out.to_parquet(artifact, index=False)
            manifest[uid] = {"status": "done",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": str(artifact)}
            n_ok += 1
        except Exception as e:
            manifest[uid] = {"status": "failed",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": None, "error": repr(e)}
            n_fail += 1
        save_manifest(scope, manifest)
    print(f"[{scope}] done={n_ok}  skipped={n_skip}  failed={n_fail}")
    if n_fail:
        print("failed units:",
              {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"})
    return manifest


def load_cached_units(scope: str, unit_ids: list) -> pd.DataFrame:
    """Analysis input: reads ONLY cached per-unit parquets, never raw npz."""
    frames = [pd.read_parquet(_unit_path(scope, str(u)))
              for u in unit_ids if _unit_path(scope, str(u)).exists()]
    return pd.concat(frames, ignore_index=True)


## 2. Scope A — within/cross-class tolerance, eyeball check

10 videos each of 3 sampled classes: a small-n, human-legible version of the
tolerance idea before committing to the full per-feature ranking at Scope B —
does a feature's value stay close across repeats of the same sign, and does
that "close" range collide with another sign's?

In [ ]:
# ============================================================
# Scope A: sample 3 classes x 10 videos, compute descriptors (small n -> no chunking)
# ============================================================
scope_a_classes = sample_classes(train_csv, N_SCOPE_A_CLASSES, seed=SEED)
print("Scope A classes:", scope_a_classes)

rng = np.random.default_rng(SEED)
scope_a_rows = []
for sign in scope_a_classes:
    pool = train_csv[train_csv["sign"] == sign]
    take = pool.sample(n=min(N_SCOPE_A_VIDEOS, len(pool)), random_state=SEED)
    scope_a_rows.append(take)
scope_a_df = pd.concat(scope_a_rows, ignore_index=True)


def _scope_a_unit(uid: str) -> pd.DataFrame:
    row = scope_a_df[scope_a_df["uid"] == uid].iloc[0]
    return descriptors_for_rows(row.to_frame().T)


process_units("scope_a", scope_a_df["uid"].tolist(), _scope_a_unit, desc="scope A")
scope_a_desc = load_cached_units("scope_a", scope_a_df["uid"].tolist())
print(scope_a_desc.shape)


In [ ]:
# ============================================================
# Scope A: nearest-neighbor margins + a couple of named features, concretely
# ============================================================
feature_cols = [c for c in scope_a_desc.columns if c not in ("uid", "sign")]
X_a = scope_a_desc[feature_cols].to_numpy(dtype=np.float32)
y_a = scope_a_desc["sign"].to_numpy()

nn_a = disc.nearest_neighbor_margins(X_a, y_a)
print(nn_a.groupby(nn_a["class"])["margin"].describe())

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(nn_a["nearest_same_class"], bins=15, alpha=0.6, label="nearest same-class")
ax.hist(nn_a["nearest_cross_class"], bins=15, alpha=0.6, label="nearest cross-class")
ax.set_xlabel("Euclidean distance (full descriptor space)")
ax.legend()
ax.set_title("Scope A: same-class vs cross-class nearest-neighbor distance")
plt.tight_layout()
plt.show()

# A couple of named angle features, concretely: per-class mean +/- TOLERANCE_K*std
example_features = ["angle_R_index_mcp_flex_mean", "angle_L_elbow_angle_mean"]
bands_a = disc.tolerance_bands(X_a, y_a,
                               [c for c in feature_cols], k=TOLERANCE_K)
print(bands_a[bands_a["feature"].isin(example_features)]
      .sort_values(["feature", "class"]).to_string(index=False))


## 3. Scope B — 15 sampled classes: F-ratio, tolerance-band overlap, probe

The main discriminability pass. Global scope (all 250 classes / 94,477 videos)
is **not** run here — filed as a follow-up (§5).

In [ ]:
# ============================================================
# Scope B: sample classes, chunk their videos, compute descriptors resumably
# ============================================================
scope_b_classes = sample_classes(train_csv, N_SCOPE_B_CLASSES, seed=SEED)
scope_b_df = (train_csv[train_csv["sign"].isin(scope_b_classes)]
              .sort_values("uid").reset_index(drop=True))
print(f"Scope B: {len(scope_b_classes)} classes, {len(scope_b_df):,} videos")

chunks = [scope_b_df.iloc[i:i + CHUNK_SIZE] for i in range(0, len(scope_b_df), CHUNK_SIZE)]
chunk_ids = [str(i) for i in range(len(chunks))]


def _scope_b_unit(chunk_id: str) -> pd.DataFrame:
    return descriptors_for_rows(chunks[int(chunk_id)])


process_units("scope_b", chunk_ids, _scope_b_unit, desc="scope B")
scope_b_desc = load_cached_units("scope_b", chunk_ids)
print(scope_b_desc.shape)


In [ ]:
# ============================================================
# Scope B: F-ratio, tolerance-band overlap, probe classifier
# ============================================================
feature_cols_b = [c for c in scope_b_desc.columns if c not in ("uid", "sign")]
X_b, feature_names_b = disc.descriptor_matrix(
    scope_b_desc[feature_cols_b].to_dict("records")
)
y_b = scope_b_desc["sign"].to_numpy()

f_tbl = disc.f_ratio(X_b, y_b, feature_names_b)
bands_b = disc.tolerance_bands(X_b, y_b, feature_names_b, k=TOLERANCE_K)
overlap_tbl = disc.band_overlap(bands_b)
probe = disc.probe_classifier(X_b, y_b, seed=SEED, val_frac=VAL_FRAC)

ranked = f_tbl.merge(overlap_tbl, on="feature")
# A landmark that's simply never detected in this sample (e.g. a pose point
# out of frame) is CONSTANT across every class -> zero-width tolerance band
# -> mean_overlap == 0.0 by construction, even though it carries zero class
# information (F == 0 too). That's a degenerate "perfect" score, not a real
# one, so overlap rankings below always gate on F > 0 first.
informative = ranked[ranked["F"] > 0]
print("Probe (all features):", probe)
print(f"{len(ranked) - len(informative)} of {len(ranked)} features are constant "
      f"(F=0, e.g. never-detected landmarks) - excluded from overlap ranking below")
print(ranked.sort_values("F", ascending=False).head(20).to_string(index=False))
print(informative.sort_values("mean_overlap").head(20).to_string(index=False))


In [ ]:
# ============================================================
# Scope B: breakdown by feature TYPE (angle/position/speed/jitter/rolling-var/
# relational/detection_rate) and by landmark REGION (face/pose/hands)
# ============================================================
def feature_type(name: str) -> str:
    if name.startswith("angle_"):
        return "angle"
    if name.startswith("relational_"):
        return "relational"
    if name.startswith("jitter_rms_"):
        return "jitter"
    if name.startswith("rolling_var_speed_"):
        return "rolling_var"
    if "_detection_rate" in name:
        return "detection_rate"
    if "_speed_" in name:
        return "speed"
    if name in ("n_frames", "valid_frame_frac"):
        return "meta"
    return "position"


def feature_region(name: str) -> str:
    for region in ("face", "left_hand", "pose", "right_hand"):
        if name.startswith(f"{region}_") or f"_{region}_" in name:
            return region
    return "cross-region"  # angles/relational span two regions (e.g. arm, inter-hand)


ranked["type"] = ranked["feature"].map(feature_type)
ranked["region"] = ranked["feature"].map(feature_region)

print("Mean F by feature type:")
print(ranked.groupby("type")["F"].mean().sort_values(ascending=False))
print()
print("Mean tolerance-band overlap by feature type (lower = better):")
print(ranked.groupby("type")["mean_overlap"].mean().sort_values())
print()
print("Mean F by region:")
print(ranked.groupby("region")["F"].mean().sort_values(ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ranked.groupby("type")["F"].mean().sort_values().plot.barh(ax=axes[0], title="Mean F by feature type")
ranked.groupby("type")["mean_overlap"].mean().sort_values(ascending=False).plot.barh(
    ax=axes[1], title="Mean tolerance-band overlap by type (lower better)")
plt.tight_layout()
plt.show()


## 4. Verdict — top discriminative features & joints

Fill in after running: which feature type (angle vs position vs velocity vs
distance vs jitter) ranks highest by F-ratio and by tolerance-band overlap:
agree or disagree with each other, and with the region-level ordering from
`motion-energy`/`subset-comparison`/`landmark-importance` (hands > pose >
face)? Which specific joint angles (if any) rank above the best raw-position
features — i.e. did engineering angles actually buy anything over xyz?

In [ ]:
# ============================================================
# Verdict tables (fill in narrative above after running)
# ============================================================
print("Top 15 by F-ratio, angle features only:")
print(ranked[ranked["type"] == "angle"].sort_values("F", ascending=False).head(15)
      .to_string(index=False))
print()
print("Top 15 by tightest tolerance band (lowest mean_overlap), any type:")
print("(gated on F > 0 -- see the note in the Scope B scoring cell: a constant,")
print(" never-detected landmark has a trivially zero-width band and would")
print(" otherwise look like a perfect, but meaningless, top result)")
print(ranked[ranked["F"] > 0].sort_values("mean_overlap").head(15).to_string(index=False))


## 5. Follow-ups & what this notebook deliberately does NOT do

- **Global scope (all 94,477 videos / 250 classes)** — not run here. Same
  staging as `subset-comparison` (scope A -> B before committing to the
  ~50-minute global pass): validate the feature set and ranking approach at
  15 classes first.
- **No trained-model ablation** — this only measures input information
  content (mirrors `subset-comparison`'s probe caveat), same as before; a
  winning feature subset from here would need a trained-model confirmation
  the way §3.0's probe findings fed §3.1's ablations.
- **Feature count vs sample count** — Scope B's probe runs on ~5,000 features
  over a few thousand videos; `LogisticRegression`'s L2 default handles this
  the same way it did for `subset-comparison`'s 543x6=3,258-dim descriptors,
  but it means the probe accuracy number is not directly comparable to a
  trained sequence model's.
- **Tolerance-band width (`TOLERANCE_K`)** is a single global choice (1.5).
  Worth an explicit sensitivity check before treating `mean_overlap` rankings
  as final — a follow-up, not done in this pass.
